In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import re
import os
from itertools import cycle


animal = ["крыса", "бык", "тигр", "кролик", "дракон", "змея", "лошадь", "коза", "обезьяна", "петух", "собака", "свинья"]
stihiya = ["дерево", "дерево", "огонь", "огонь", "почва", "почва",  "металл",  "металл", "вода", "вода"]
inYan = ["ян", "инь"]

visYear = [1920, 1924, 1928, 1932, 1936, 1940, 1944, 1948, 1952, 1956, 1960, 1964, 1968, 
            1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 
            2024, 2028, 2032, 2036, 2040, 2044, 2048, 2052]

moonPalace = dict({1: [1920, 1942, 0, 1987, 2009, 2032], 2: [0, 1943, 1965, 1988, 2010, 0], 
    3: [1921, 1944, 1966, 0, 2011, 2033], 4: [1922, 0, 1967, 1989, 2012, 2034], 
    5: [1923, 1945, 1968, 1990, 0, 2035], 6: [1924, 1946, 0, 1991, 2013, 2036], 
    7: [0, 1947, 1969, 1992, 2014, 0], 8: [1925, 1948, 1970, 0, 2015, 2037], 
    9: [1926, 0, 1971, 1993, 2016, 2038], 10: [1927, 1949, 1972, 1994, 0, 2039], 
    11: [1928, 1950, 0, 1995, 2017, 2040], 12: [0, 1951, 1973, 1996, 2018, 0], 
    13: [1929, 1952, 1974, 0, 2019, 2041], 14: [1930, 0, 1975, 1997, 2020, 2042], 
    15: [1931, 1953, 1976, 1998, 0, 2043], 16: [1932, 1954, 0, 1999, 2021, 2044], 
    17: [0, 1955, 1977, 2000, 2022, 0], 18: [1933, 1956, 1978, 0, 2023, 2045], 
    19: [1934, 0, 1979, 2001, 2024, 2046], 20: [1935, 1957, 1980, 2002, 0, 2047], 
    21: [1936, 1958, 0, 2003, 2025, 2048], 22: [0, 1959, 1981, 2004, 2026, 0], 
    23: [1937, 1960, 1982, 0, 2027, 2049], 24: [1938, 0, 1983, 2005, 2028, 2050], 
    25: [1939, 1961, 1984, 2006, 0, 2051], 26: [1940, 1962, 0, 2007, 2029, 2052], 
    27: [0, 1963, 1985, 2008, 2030, 0], 28: [1941, 1964, 1986, 0, 2031, 2053]})

secStep = {1: 27,
            2: 2,
            3: 2,
            4: 5,
            5: 7,
            6: 10,
            7: 12,
            8: 15,
            9: 18,
            10: 20,
            11: 23,
            12: 25}


man = ["Liv.1", "Liv.4", "Liv.3", "Gb.37/Liv.3", "Liv.5/Gb.40", "Liv.2", "Liv.8", 
       "Kid.1", "Kid.7", "Kid.3", "Bl.58/Kid.3", "Kid.4/Bl.64", "Kid.2", "Kid.10", 
       "Lu.11", "Lu.8", "Lu.9", "Co.6/Lu.9", "Co.4/Lu.7", "Lu.10", "Lu.5", 
       "Ht.9/Hg.9", "Ht.4/Hg.5", "Ht.7/Hg.7", "Si.7/Ht.7/Hg.7", 
       "Ht.5/Hg.6/Si.4", "Ht.8/Hg.8", "Ht.3/ Hg.3"]

woman = ["Gb.41", "Gb.44", "Gb.34", "Gb.37/Liv.3", "Liv.5/Gb.40", "Gb.38", "Gb.43", 
       "Bl.65", "Bl.67", "Bl.40", "Bl.58/Kid.3", "Kid.4/Bl.64", "Bl.60", "Bl.66", 
       "Co.3", "Co.1", "Co.11", "Co.6/Lu.9", "Co4/Lu.7", "Co.5", "Co.2", 
       "Si.3", "Si.1", "Si.8", "Si.7/Ht.7/Hg.7", "Ht.5/Hg.6/Si.4", "Si.5", "Si.2"]

sky = {'甲': ':green[甲]',
        '乙': ':green[乙]',
        '丙': ':red[丙]',
        '丁': ':red[丁]',
        '戊': ':orange[戊]',
        '己': ':orange[己]',
        '庚': ':darkgray[庚]',
        '辛': ':darkgray[辛]',
        '壬': ':blue[壬]',
        '癸': ':blue[癸]'}

earth = {'子': ':blue[子]',
        '丑': ':orange[丑]',
        '寅': ':green[寅]',
        '卯': ':green[卯]',
        '辰': ':orange[辰]',
        '巳': ':red[巳]',
        '午': ':red[午]',
        '未': ':orange[未]',
        '申': ':darkgray[申]',
        '酉': ':darkgray[酉]',
        '戌': ':orange[戌]',
        '亥': ':blue[亥]'}

colorDict = {'甲':'green', '乙':'green', '丙':'red', '丁':'red', '戊':'orange', '己':'orange', '庚':'grey', '辛':'grey', '壬':'blue', '癸':'blue',
                '子':'blue', '丑':'orange', '寅':'green', '卯':'green', '辰':'orange', '巳':'red', '午':'red', '未':'orange', '申':'grey', '酉':'grey', '戌':'orange', '亥':'blue'}

colorDictEarth = {'子':'blue', '丑':'orange', '寅':'green', '卯':'green', '辰':'orange', '巳':'red', '午':'red', '未':'orange', '申':'grey', '酉':'grey', '戌':'orange', '亥':'blue'}

def read_files():       
    cities = pd.read_csv("data/cities.csv")
    earth_legs = pd.read_csv("data/earth_legs.csv")
    sky_hands = pd.read_csv("data/sky_hands.csv")
    planets = pd.read_csv("data/planets.csv")
    moon_palace_df = pd.read_csv("data/moon_palace.csv")
    calendar = pd.read_csv("data/calendar.csv")
    cicle = pd.read_csv("data/cicle.csv")
    seasons = pd.read_csv("data/seasons.csv")
    veto = pd.read_csv("data/veto.csv")
    return cities, earth_legs, sky_hands, planets, moon_palace_df, calendar, cicle, seasons, veto

dolgoletie = {
    'ду-май':'жень-май',
    'чун-май':'дай-май',
    'чонг-май':'дай-май',
    'инь-цзяо':'ян-цзяо',
    'инь-вэй':'ян-вэй',
    'жень-май':'ду-май',
    'жэнь-май':'ду-май',
    'дай-май':'чонг-май',
    'ян-цзяо':'инь-цзяо',
    'ян-вэй':'инь-вэй',
}

skydoc = {
    'ду-май':'ян-цзяо',
    'чун-май':'инь-вэй',
    'чонг-май':'инь-вэй',
    'инь-цзяо':'жень-май',
    'ян-вэй':'дай-май',
    'ян-цзяо':'ду-май',
    'инь-вэй':'чонг-май',
    'жень-май':'инь-цзяо',
    'жэнь-май':'инь-цзяо',
    'дай-май':'ян-вэй',
}

birthqi = {
    'ду-май':'инь-цзяо',
    'чун-май':'ян-вэй',
    'чонг-май':'ян-вэй',
    'жень-май':'ян-цзяо',
    'жэнь-май':'ян-цзяо',
    'инь-вэй':'дай-май',
    'инь-цзяо':'ду-май',
    'ян-вэй':'чонг-май',
    'ян-цзяо':'жень-май',
    'дай-май':'инь-вэй',
}

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import re
import os
from itertools import cycle


animal = ["крыса", "бык", "тигр", "кролик", "дракон", "змея", "лошадь", "коза", "обезьяна", "петух", "собака", "свинья"]
stihiya = ["дерево", "дерево", "огонь", "огонь", "почва", "почва",  "металл",  "металл", "вода", "вода"]
inYan = ["ян", "инь"]

visYear = [1920, 1924, 1928, 1932, 1936, 1940, 1944, 1948, 1952, 1956, 1960, 1964, 1968, 
            1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 
            2024, 2028, 2032, 2036, 2040, 2044, 2048, 2052]

moonPalace = dict({1: [1920, 1942, 0, 1987, 2009, 2032], 2: [0, 1943, 1965, 1988, 2010, 0], 
    3: [1921, 1944, 1966, 0, 2011, 2033], 4: [1922, 0, 1967, 1989, 2012, 2034], 
    5: [1923, 1945, 1968, 1990, 0, 2035], 6: [1924, 1946, 0, 1991, 2013, 2036], 
    7: [0, 1947, 1969, 1992, 2014, 0], 8: [1925, 1948, 1970, 0, 2015, 2037], 
    9: [1926, 0, 1971, 1993, 2016, 2038], 10: [1927, 1949, 1972, 1994, 0, 2039], 
    11: [1928, 1950, 0, 1995, 2017, 2040], 12: [0, 1951, 1973, 1996, 2018, 0], 
    13: [1929, 1952, 1974, 0, 2019, 2041], 14: [1930, 0, 1975, 1997, 2020, 2042], 
    15: [1931, 1953, 1976, 1998, 0, 2043], 16: [1932, 1954, 0, 1999, 2021, 2044], 
    17: [0, 1955, 1977, 2000, 2022, 0], 18: [1933, 1956, 1978, 0, 2023, 2045], 
    19: [1934, 0, 1979, 2001, 2024, 2046], 20: [1935, 1957, 1980, 2002, 0, 2047], 
    21: [1936, 1958, 0, 2003, 2025, 2048], 22: [0, 1959, 1981, 2004, 2026, 0], 
    23: [1937, 1960, 1982, 0, 2027, 2049], 24: [1938, 0, 1983, 2005, 2028, 2050], 
    25: [1939, 1961, 1984, 2006, 0, 2051], 26: [1940, 1962, 0, 2007, 2029, 2052], 
    27: [0, 1963, 1985, 2008, 2030, 0], 28: [1941, 1964, 1986, 0, 2031, 2053]})

secStep = {1: 27,
            2: 2,
            3: 2,
            4: 5,
            5: 7,
            6: 10,
            7: 12,
            8: 15,
            9: 18,
            10: 20,
            11: 23,
            12: 25}


man = ["Liv.1", "Liv.4", "Liv.3", "Gb.37/Liv.3", "Liv.5/Gb.40", "Liv.2", "Liv.8", 
       "Kid.1", "Kid.7", "Kid.3", "Bl.58/Kid.3", "Kid.4/Bl.64", "Kid.2", "Kid.10", 
       "Lu.11", "Lu.8", "Lu.9", "Co.6/Lu.9", "Co.4/Lu.7", "Lu.10", "Lu.5", 
       "Ht.9/Hg.9", "Ht.4/Hg.5", "Ht.7/Hg.7", "Si.7/Ht.7/Hg.7", 
       "Ht.5/Hg.6/Si.4", "Ht.8/Hg.8", "Ht.3/ Hg.3"]

woman = ["Gb.41", "Gb.44", "Gb.34", "Gb.37/Liv.3", "Liv.5/Gb.40", "Gb.38", "Gb.43", 
       "Bl.65", "Bl.67", "Bl.40", "Bl.58/Kid.3", "Kid.4/Bl.64", "Bl.60", "Bl.66", 
       "Co.3", "Co.1", "Co.11", "Co.6/Lu.9", "Co4/Lu.7", "Co.5", "Co.2", 
       "Si.3", "Si.1", "Si.8", "Si.7/Ht.7/Hg.7", "Ht.5/Hg.6/Si.4", "Si.5", "Si.2"]

sky = {'甲': ':green[甲]',
        '乙': ':green[乙]',
        '丙': ':red[丙]',
        '丁': ':red[丁]',
        '戊': ':orange[戊]',
        '己': ':orange[己]',
        '庚': ':darkgray[庚]',
        '辛': ':darkgray[辛]',
        '壬': ':blue[壬]',
        '癸': ':blue[癸]'}

earth = {'子': ':blue[子]',
        '丑': ':orange[丑]',
        '寅': ':green[寅]',
        '卯': ':green[卯]',
        '辰': ':orange[辰]',
        '巳': ':red[巳]',
        '午': ':red[午]',
        '未': ':orange[未]',
        '申': ':darkgray[申]',
        '酉': ':darkgray[酉]',
        '戌': ':orange[戌]',
        '亥': ':blue[亥]'}

colorDict = {'甲':'green', '乙':'green', '丙':'red', '丁':'red', '戊':'orange', '己':'orange', '庚':'grey', '辛':'grey', '壬':'blue', '癸':'blue',
                '子':'blue', '丑':'orange', '寅':'green', '卯':'green', '辰':'orange', '巳':'red', '午':'red', '未':'orange', '申':'grey', '酉':'grey', '戌':'orange', '亥':'blue'}

colorDictEarth = {'子':'blue', '丑':'orange', '寅':'green', '卯':'green', '辰':'orange', '巳':'red', '午':'red', '未':'orange', '申':'grey', '酉':'grey', '戌':'orange', '亥':'blue'}

def read_files():       
    cities = pd.read_csv("data/cities.csv")
    earth_legs = pd.read_csv("data/earth_legs.csv")
    sky_hands = pd.read_csv("data/sky_hands.csv")
    planets = pd.read_csv("data/planets.csv")
    moon_palace_df = pd.read_csv("data/moon_palace.csv")
    calendar = pd.read_csv("data/calendar.csv")
    cicle = pd.read_csv("data/cicle.csv")
    seasons = pd.read_csv("data/seasons.csv")
    veto = pd.read_csv("data/veto.csv")
    return cities, earth_legs, sky_hands, planets, moon_palace_df, calendar, cicle, seasons, veto

dolgoletie = {
    'ду-май':'жень-май',
    'чун-май':'дай-май',
    'чонг-май':'дай-май',
    'инь-цзяо':'ян-цзяо',
    'инь-вэй':'ян-вэй',
    'жень-май':'ду-май',
    'жэнь-май':'ду-май',
    'дай-май':'чонг-май',
    'ян-цзяо':'инь-цзяо',
    'ян-вэй':'инь-вэй',
}

skydoc = {
    'ду-май':'ян-цзяо',
    'чун-май':'инь-вэй',
    'чонг-май':'инь-вэй',
    'инь-цзяо':'жень-май',
    'ян-вэй':'дай-май',
    'ян-цзяо':'ду-май',
    'инь-вэй':'чонг-май',
    'жень-май':'инь-цзяо',
    'жэнь-май':'инь-цзяо',
    'дай-май':'ян-вэй',
}

birthqi = {
    'ду-май':'инь-цзяо',
    'чун-май':'ян-вэй',
    'чонг-май':'ян-вэй',
    'жень-май':'ян-цзяо',
    'жэнь-май':'ян-цзяо',
    'инь-вэй':'дай-май',
    'инь-цзяо':'ду-май',
    'ян-вэй':'чонг-май',
    'ян-цзяо':'жень-май',
    'дай-май':'инь-вэй',
}




################################################################################################





cities, earth_legs, sky_hands, planets, moon_palace_df, calendar, cicle, seasons, veto = read_files()


calendar['date'] = pd.to_datetime(calendar['date'])

needed_channel = "Дай-май"
needed_time = ['辰', '巳', '午', '未']
needed_in_yan_day = "ян"
needed_methods = ["Продление жизни", "Небесный лекарь", "Порождающая ци"]
needed_method = needed_methods[1]
needed_pair = []
if needed_method == needed_methods[1]:
    needed_pair = [needed_channel, skydoc[needed_channel.lower()].capitalize()]
elif needed_method == needed_methods[2]:
    needed_pair = [needed_channel, birthqi[needed_channel.lower()].capitalize()]
else:
    needed_pair = [needed_channel, dolgoletie[needed_channel.lower()].capitalize()]

week_predictions = {"our_date":[],
                    "day_iero":[],
                    "in_yan_day":[],}

for n in range(7):
    our_date = "19.10.2025"
    our_date = datetime.strptime(our_date, "%d.%m.%Y") + timedelta(days=n)
    if our_date:
        try:
            # our_date = vis_date = re.sub('\D', '.', our_date)
            # our_date = our_date.split('.')
            d = int(our_date.day)
            m = int(our_date.month)
            y = int(our_date.year)
            our_date = date(y, m, d)
            week_predictions["our_date"].append(our_date)

        except:
            print("Некорректная дата, попробуйте снова!")

table = f'''<table>
                <tr>
                    <th> День </th>
                    <th> Месяц </th>
                    <th> Год </th>
                </tr>
                <tr>
                    <td>{highlight_words(birth_day)}</td>
                    <td>{highlight_words(birth_mon)}</td>
                    <td>{highlight_words(birth_yea)}</td>
                </tr>
                <tr>
                    <td>{highlight_words(day_ier)}</td>
                    <td>{highlight_words(month_ier)}</td>
                    <td>{highlight_words(year_ier)}</td>
                </tr>                                   
            </table>'''

for our_date in week_predictions["our_date"]:
    if pd.to_datetime(our_date) < pd.to_datetime(seasons.loc[2, str(our_date.year)]):
        year_o = calendar[calendar['date']==pd.to_datetime(pd.to_datetime(our_date)-timedelta(days=51))]['years'].values[0]
    else:
        year_o = calendar[calendar['date']==pd.to_datetime(our_date)]['years'].values[0]

    mo = calendar[calendar['date']==pd.to_datetime(our_date)]['months'].values[0]
    if pd.to_datetime(our_date) < pd.to_datetime(seasons[seasons['Месяц']==mo.split()[0]][str(pd.to_datetime(our_date).year)].values[0]):
        month_o = calendar[calendar['date']==pd.to_datetime(pd.to_datetime(our_date)-timedelta(days=21))]['months'].values[0]
    else:
        month_o = calendar[calendar['date']==pd.to_datetime(our_date)]['months'].values[0]

    day_o = calendar[calendar['date']==pd.to_datetime(our_date)]['days'].values[0]
    day = cicle[cicle["Название_calendar"] == day_o]["Название_Русский"].values[0]
    day_iero = cicle[cicle["Название_calendar"] == day_o]["Иероглиф"].values[0]
    week_predictions["day_iero"].append(day_iero)
    month_iero = cicle[cicle["Название_calendar"] == month_o]["Иероглиф"].values[0]
    year_iero = cicle[cicle["Название_calendar"] == year_o]["Иероглиф"].values[0]
    
    in_yan_day = cicle[cicle['Название_calendar'] == day_o]['инь_ян'].values[0]
    week_predictions["in_yan_day"].append(in_yan_day)


######### ФЭЙ ТЭН БА ФА ##########
    feitenbafa = pd.read_csv("data/feitenbafa.csv")
    for_feitenbafa = pd.read_csv("data/for_feitenbafa.csv")
    day_predictions = feitenbafa.merge(for_feitenbafa.rename(columns={"Иероглиф":day_iero[0]}))
    feitenbafa_day = day_predictions[[day_iero[0], 'Иероглиф',	'Время',	'Канал',	'Точки']]
    print(our_date, "\nФЭЙ ТЭН БА ФА\n___________________________")
    for row in feitenbafa_day.index:
        if (feitenbafa_day.loc[row, 'Канал'] == needed_channel):
            dict_of_row = feitenbafa_day.iloc[row].to_dict()
            print(dict_of_row['Время'])
        
######### ТАЙ ЯН БА ФА ##########
    if in_yan_day == needed_in_yan_day:
        print(needed_method)
        list_tai = os.listdir("data/tai_yan_ba_fa_2/")
        for l in list_tai:
            if day_iero[0] in l:
                file=re.findall(f'(\w*{day_iero[0]}\w*.csv)', l)

        df = pd.read_csv(f"data/tai_yan_ba_fa_2/{file[0]}")

        for i in df.columns[2:]:
            df[i] = df[i].apply(lambda x: x.split()[0] if len(x)>3 else x)
        
        for i in df.index:
            row = df.iloc[i].to_list()
            if (needed_pair[0] in row[2:]) and (needed_pair[1] in row[2:]):
                print(row[1:])
    print("\n___________________________")

2025-10-19 
ФЭЙ ТЭН БА ФА
___________________________
23:00 - 01:00
19:00 - 21:00

___________________________
2025-10-20 
ФЭЙ ТЭН БА ФА
___________________________
15:00 - 17:00
Небесный лекарь
['01.30 - 03.00', 'Ян-вэй', 'Инь-вэй', 'Чонг-май', 'Дай-май']
['03.00 - 4.30', 'Ян-вэй', 'Инь-вэй', 'Чонг-май', 'Дай-май']
['04.30 - 5.00', 'Чонг-май', 'Дай-май', 'Инь-вэй', 'Ян-вэй']
['05.00 - 7.00', 'Чонг-май', 'Дай-май', 'Инь-вэй', 'Ян-вэй']
['07.00 - 7.30', 'Чонг-май', 'Дай-май', 'Инь-вэй', 'Ян-вэй']
['13.30 - 15.00', 'Инь-вэй', 'Ян-вэй', 'Чонг-май', 'Дай-май']
['19.00 - 19.30', 'Дай-май', 'Чонг-май', 'Инь-вэй', 'Ян-вэй']

___________________________
2025-10-21 
ФЭЙ ТЭН БА ФА
___________________________
11:00 - 13:00

___________________________
2025-10-22 
ФЭЙ ТЭН БА ФА
___________________________
7:00 - 9:00
Небесный лекарь
['01.30 - 03.00', 'Ян-вэй', 'Инь-вэй', 'Чонг-май', 'Дай-май']
['05.00 - 7.00', 'Чонг-май', 'Дай-май', 'Инь-вэй', 'Ян-вэй']
['13.30 - 15.00', 'Инь-вэй', 'Ян-вэй', 'Чонг

In [266]:
list_tai = ['丁壬', '丙辛', '乙庚', '戊癸', '甲己']

dict_tai = {"丁壬":"first",
"丙辛":"second",
"乙庚":"trid",
"戊癸":"fourth",
"甲己":"fiveth",}

for l in list_tai:
    tai_yan_ba_fa = pd.read_csv(f"data/tai_yan_ba_fa/{l}.csv")

    for i in tai_yan_ba_fa.index:
        for j in tai_yan_ba_fa.columns:
            if i%2==0:
                tai_yan_ba_fa.iloc[i, int(j)] = tai_yan_ba_fa.iloc[i, int(j)] + " " + tai_yan_ba_fa.iloc[i+1, int(j)]


    df = tai_yan_ba_fa.drop(tai_yan_ba_fa.index[range(1,42, 2)], axis=0).reset_index(drop=True)

    df.iloc[:,0] = df.iloc[:,0].str.strip()
    df.iloc[:,1] = df.iloc[:,1].str.strip()
    df.iloc[:,2] = df.iloc[:,2].str.strip()
    df.iloc[:,3] = df.iloc[:,3].str.strip()
    df.iloc[:,4] = df.iloc[:,4].str.strip()
    df.iloc[:,5] = df.iloc[:,5].str.strip()

    for i in df.index:
        if  len(df.loc[i, '4']) == 0:
            df.loc[i, '4'] = df.loc[i, '4'] + " "
    for i in df.index:
        if  len(df.loc[i, '5']) == 0:
            df.loc[i, '5'] = df.loc[i, '5'] + " "
    # for i in df.columns[2:]:
    #     df[i] = df[i].apply(lambda x: x.split()[0] if len(x)>3 else x)
    # for i in df.columns:
    #     df[i] = df[i].str.strip()
    # for j in df.index:
    #     if len(df.loc[j, i])>1:
    #         df.loc[j, i] = df.loc[j, i].strip()
    # df.to_csv(f"data/tai_yan_ba_fa/{dict_tai[l]}.csv", index=False)
    # display(df)

In [ ]:
hronopunktura

In [284]:
string = 'email@хронопунктура.рф'
string

'email@хронопунктура.рф'

In [289]:
import idna 
local_part, domain = string.split('@')
# Конвертируем домен в Punycode
domain_ascii = idna.encode(domain).decode('ascii')
domain_ascii

'xn--80atiadbhegtidq.xn--p1ai'

In [280]:
enc = idna.encode(string)#.decode('ascii')
enc

b'xn--80atiadbhegtidq.xn--p1ai'

In [283]:
enc.decode("ascii")

'xn--80atiadbhegtidq.xn--p1ai'